Lib sẽ được để khai báo đầu tiên

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
# !pip install kagglehub
import kagglehub
import zipfile
import os
import shutil
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import joblib
import random
from sklearn.cluster import KMeans

Lấy dữ liệu có chọn lọc Dataset 20BN-JESTER từ kaggle

In [ ]:

path = kagglehub.dataset_download("toxicmender/20bn-jester")
print("Path to dataset files:", path)
TARGET_LABELS = [
    'Swiping Left', 
    'Swiping Right', 
    'Stop Sign', 
    'Thumb Up', 
    'No gesture'
]
zip_path = '/content/20bn-jester-v1.zip'
csv_path = '/content/jester-v1-train.csv'
extract_path = '/content/jester_filtered'
if not os.path.exists(extract_path):
    os.makedirs(extract_path)
print(f"Dữ liệu gốc nằm tại: {path}")
all_files = []
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".csv"):
            print(f" tìm thấy file CSV: {os.path.join(root, file)}")
            csv_path = os.path.join(root, file)
def filter_and_copy_data(source_path, csv_file, target_labels, destination):
    df = pd.read_csv(csv_file, sep=';', header=None, names=['video_id', 'label'])
    filtered_df = df[df['label'].isin(target_labels)]
    target_ids = set(filtered_df['video_id'].astype(str).tolist())
    print(f"Bắt đầu lọc {len(target_ids)} video cho các nhãn: {target_labels}")
    video_root = ""
    for root, dirs, files in os.walk(source_path):
        if any(d.isdigit() for d in dirs):
            video_root = root
            break
    count = 0
    for v_id in tqdm(target_ids):
        src_v_path = os.path.join(video_root, v_id)
        dst_v_path = os.path.join(destination, v_id)
        if os.path.exists(src_v_path):
            shutil.move(src_v_path, dst_v_path)
            count += 1
    print(f"Đã đưa {count} video về {destination}")
    return filtered_df

filtered_metadata = filter_and_copy_data(path, csv_path, TARGET_LABELS, extract_path)
# Kagglehub path
base_path = '/root/.cache/kagglehub/datasets/toxicmender/20bn-jester/versions/3'
csv_path = os.path.join(base_path, 'Train.csv')
train_folder = os.path.join(base_path, 'Train') 
df = pd.read_csv(csv_path, sep=',')

TARGET_LABELS = [
    'Swiping Left', 
    'Swiping Right', 
    'Stop Sign', 
    'Thumbs Up', 
    'No gesture'
]
filtered_df = df[df['label'].isin(TARGET_LABELS)]
target_ids = set(filtered_df['video_id'].astype(str).tolist())
print(f"Tìm thấy {len(target_ids)} video phù hợp!")
print("Các nhãn vừa lọc:", filtered_df['label'].unique())

Train KMean

In [ ]:
# thực thi đoạn mã để lấy API từ google
# Invoke-WebRequest -Uri "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task" -OutFile "hand_landmarker.task"
base_options = python.BaseOptions(
    model_asset_path='hand_landmarker.task',
    delegate=python.BaseOptions.Delegate.CPU
)
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=1)
detector = vision.HandLandmarker.create_from_options(options)

def get_landmarks(img_path):
    cv2_img = cv2.imread(img_path)
    if cv2_img is None: return None
    
    # Chuyển đổi định dạng ảnh
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(cv2_img, cv2.COLOR_BGR2RGB))
    result = detector.detect(mp_image)
    
    if result.hand_landmarks:
        # Chuyển sang NumPy array (21, 3)
        coords = np.array([[lm.x, lm.y, lm.z] for lm in result.hand_landmarks[0]])
        # 1. Dịch tâm: Cổ tay (Wrist - Index 0) về gốc tọa độ (0,0,0)
        relative_coords = coords - coords[0]
        # 2. Chuẩn hóa tỷ lệ: Chia cho khoảng cách lớn nhất để triệt tiêu độ to/nhỏ của tay
        max_val = np.abs(relative_coords).max()
        if max_val > 0:
            relative_coords /= max_val
        return relative_coords.flatten() # Trả về vector phẳng 63 chiều
    return None
data_root = '/content/jester_filtered'
all_video_ids = [d for d in os.listdir(data_root) if os.path.isdir(os.path.join(data_root, d))]
samples = []
print("Đang trích xuất mẫu landmarks để train K-means...")
pbar = tqdm(total=500)

while len(samples) < 500:
    v_id = random.choice(all_video_ids)
    v_path = os.path.join(data_root, v_id)
    frames = [f for f in os.listdir(v_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if not frames: continue
    f_name = random.choice(frames)
    lm = get_landmarks(os.path.join(v_path, f_name))
    if lm is not None:
        samples.append(lm)
        pbar.update(1)
pbar.close()
print(f"\nĐang phân thành 32 cụm tư thế tay...")
training_data = np.array(samples)
kmeans = KMeans(n_clusters=32, n_init=10, random_state=42)
kmeans.fit(training_data)
# Lưu vào Drive 
# save_dir = '/content/drive/MyDrive/Project_DA_DS371'
# if not os.path.exists(save_dir): os.makedirs(save_dir)

# model_file = os.path.join(save_dir, 'hand_pose_kmeans.pkl')
# joblib.dump(kmeans, model_file)

# print(f"Done! Bộ từ điển tư thế tay nằm tại: {model_file}")

Số hóa dataset (chỉ cần chạy khi thực hiện lần train đầu tiên)

In [ ]:
# filtered_df.to_csv('train_filtered.csv', index=False)
# kmeans = joblib.load('/content/drive/MyDrive/Project_DA_DS371/hand_pose_kmeans.pkl')
# label_map = {label: i for i, label in enumerate(filtered_df['label'].unique())}
# def process_all_data(df, data_root, num_frames=16):
#     X_tokens = []
#     y_labels = []
#     print(f"Bắt đầu 'số hóa' {len(df)} video...")
#     for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing Videos"):
#         v_id = str(row['video_id'])
#         label_name = row['label']
#         v_path = os.path.join(data_root, v_id)
#         if not os.path.exists(v_path):
#             continue
#         frames = sorted([f for f in os.listdir(v_path) if f.endswith('.jpg')])
#         if len(frames) == 0: continue
#         if len(frames) <= num_frames:
#             indices = [i % len(frames) for i in range(num_frames)]
#         else:
#             indices = [i * len(frames) // num_frames for i in range(num_frames)]
#         video_tokens = []
#         for i in indices:
#             img_path = os.path.join(v_path, frames[i])
#             lm = get_landmarks(img_path)
#             if lm is not None:
#                 token = kmeans.predict(lm.reshape(1, -1))[0]
#                 video_tokens.append(token)
#             else:
#                 video_tokens.append(32)
#         X_tokens.append(video_tokens)
#         y_labels.append(label_map[label_name])
#     return np.array(X_tokens), np.array(y_labels)
# X_train, y_train = process_all_data(filtered_df, '/content/jester_filtered')
# np.save('/content/drive/MyDrive/Project_DA_DS371/X_train_final.npy', X_train)
# np.save('/content/drive/MyDrive/Project_DA_DS371/y_train_final.npy', y_train)

Thực hiện việc train LSTM

In [ ]:


class FastGestureDataset(Dataset):
    def __init__(self, x_path, y_path, csv_path):
        self.x_data = torch.LongTensor(np.load(x_path))
        self.y_data = torch.tensor(np.load(y_path), dtype=torch.long)
        df = pd.read_csv(csv_path)
        self.classes = sorted(df['label'].unique().tolist())

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        return self.x_data[idx], self.y_data[idx]

# --- 2. ARCHITECTURE: LSTM MODEL ---
class GestureLSTM(nn.Module):
    def __init__(self, vocab_size=33, embed_dim=128, hidden_dim=256, num_classes=4):
        super(GestureLSTM, self).__init__()
        # Tăng embed_dim và hidden_dim lên vì T4 GPU dư sức gánh
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, num_layers=2, dropout=0.3)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.embedding(x)
        # hn là hidden state của frame cuối cùng: (num_layers, batch_size, hidden_dim)
        _, (hn, _) = self.lstm(x)
        # Lấy layer cuối cùng của LSTM
        out = self.fc(hn[-1]) 
        return out

# --- 3. TRAINER
class GestureTrainer:
    def __init__(self, model, device):
        self.model = model.to(device)
        self.device = device
        self.history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    def train(self, train_loader, val_loader, epochs=50, lr=0.0005):
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.model.parameters(), lr=lr)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)
        print(f"Bắt đầu huấn luyện trên thiết bị: {self.device}")
        for epoch in range(epochs):
            self.model.train()
            t_loss, t_correct = 0, 0
            # Thanh tiến trình TQDM
            pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", unit="batch")
            for x, y in pbar:
                # Đưa dữ liệu lên GPU
                x, y = x.to(self.device), y.to(self.device)
                optimizer.zero_grad()
                out = self.model(x)
                loss = criterion(out, y)
                loss.backward()
                optimizer.step()
                t_loss += loss.item()
                t_correct += (out.argmax(1) == y).sum().item()
                pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            train_loss = t_loss / len(train_loader)
            train_acc = t_correct / len(train_loader.dataset)
            # Validation
            val_loss, val_acc = self.evaluate(val_loader, criterion)
            scheduler.step(val_loss)
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_acc'].append(val_acc)
            print(f"KQ Epoch {epoch+1}: Train Acc {train_acc:.4f} | Val Acc {val_acc:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    def evaluate(self, loader, criterion=None):
        self.model.eval()
        loss, correct = 0, 0
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(self.device), y.to(self.device)
                out = self.model(x)
                if criterion: loss += criterion(out, y).item()
                correct += (out.argmax(1) == y).sum().item()
        return loss/len(loader) if criterion else 0, correct/len(loader.dataset)

    def plot_final(self, test_loader, class_names):
        # 1. Đồ thị Loss & Accuracy
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        ax1.plot(self.history['train_loss'], label='Train'); ax1.plot(self.history['val_loss'], label='Val')
        ax1.set_title('Loss Curve'); ax1.legend()
        ax2.plot(self.history['train_acc'], label='Train'); ax2.plot(self.history['val_acc'], label='Val')
        ax2.set_title('Accuracy Curve'); ax2.legend()
        plt.show()

        # 2. Confusion Matrix
        all_preds, all_labels = [], []
        self.model.eval()
        with torch.no_grad():
            for x, y in test_loader:
                out = self.model(x.to(self.device))
                all_preds.extend(out.argmax(1).cpu().numpy())
                all_labels.extend(y.numpy())
        
        cm = confusion_matrix(all_labels, all_preds)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
        plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
        plt.show()

# --- 4. THỰC THI CHƯƠNG TRÌNH ---
# Đường dẫn file trên Drive
X_PATH = '/content/drive/MyDrive/Project_DA_DS371/X_train_final.npy'
Y_PATH = '/content/drive/MyDrive/Project_DA_DS371/y_train_final.npy'
CSV_PATH = '/content/drive/MyDrive/Project_DA_DS371/train_filtered.csv' 
full_ds = FastGestureDataset(X_PATH, Y_PATH, CSV_PATH)
train_size = int(0.8 * len(full_ds))
val_ds, train_ds = random_split(full_ds, [len(full_ds)-train_size, train_size])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True) # Tăng batch_size lên cho GPU
val_loader = DataLoader(val_ds, batch_size=64)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GestureLSTM(num_classes=len(full_ds.classes))
trainer = GestureTrainer(model, device)
trainer.train(train_loader, val_loader, epochs=50)
torch.save(model.state_dict(), '/content/drive/MyDrive/Project_DA_DS371/gesture_lstm_t4.pth')

# Kết quả cuối cùng
trainer.plot_final(val_loader, full_ds.classes)